<a href="https://colab.research.google.com/github/Farhan1Ahmed1Molla/universal-data-santizer/blob/main/case_studies_02_laptop_sales_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re

def sanitize_data(df):
    clean_df = df.copy()

    # LAYER 1: Standardize Column Headers
    clean_df.columns = (
        clean_df.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[^\w\s]', '', regex=True)
        .str.replace(r'\s+', '_', regex=True)
    )

    # LAYER 2: Drop 100% Duplicate Rows
    initial_rows = len(clean_df)
    clean_df = clean_df.drop_duplicates().reset_index(drop=True)
    dupes_removed = initial_rows - len(clean_df)

    # LAYER 3: Clean Text Columns & Convert Disguised Nulls
    fake_nulls = ['n/a', 'na', 'null', 'none', 'error', 'unknown', '-', '--', '?', 'tbd', '']

    for col in clean_df.select_dtypes(include='object').columns:
        clean_df[col] = clean_df[col].astype(str).str.strip()
        clean_df[col] = clean_df[col].apply(
            lambda x: np.nan if x.lower() in fake_nulls or x == 'nan' else x
        )

    # LAYERS 4 & 5: Auto-Date Parsing & Smart Numeric Scrubbing
    for col in clean_df.select_dtypes(include='object').columns:
        non_null_count = clean_df[col].notna().sum()
        if non_null_count == 0:
            continue

        # Layer 4: Dates
        if 'date' in col:
            clean_df[col] = pd.to_datetime(clean_df[col], errors='coerce', format='mixed')
            clean_df = clean_df.sort_values(by=col, ascending=True).reset_index(drop=True)
            continue

        # Layer 5: Numbers
        temp_num = clean_df[col].str.replace(r'[$€£₹,%]', '', regex=True).str.strip()
        converted_num = pd.to_numeric(temp_num, errors='coerce')

        if (converted_num.notna().sum() / non_null_count) >= 0.70:
            clean_df[col] = converted_num

    # FINAL OUTPUT (Must stay at the very bottom of the function)
    print(f"Headers standardized | Duplicate rows removed: {dupes_removed}")
    return clean_df

In [ ]:
import kagglehub
import os
import pandas as pd

# Download Dataset #2: Uncleaned Laptop Price Dataset
path2 = kagglehub.dataset_download("ehtishamsadiq/uncleaned-laptop-price-dataset")
csv_path2 = os.path.join(path2, os.listdir(path2)[0])

raw_laptop_df = pd.read_csv(csv_path2)

clean_laptop_df = sanitize_data(raw_laptop_df)

print("--- DATA TYPES ---")
print(clean_laptop_df.dtypes)

print("\n--- MISSING VALUES ---")
print(clean_laptop_df.isna().sum())

print("\n--- DATA SAMPLE ---")
display(clean_laptop_df.head(3))

Using Colab cache for faster access to the 'uncleaned-laptop-price-dataset' dataset.
Headers standardized | Duplicate rows removed: 29
--- DATA TYPES ---
unnamed_0           float64
company              object
typename             object
inches              float64
screenresolution     object
cpu                  object
ram                  object
memory               object
gpu                  object
opsys                object
weight               object
price               float64
dtype: object

--- MISSING VALUES ---
unnamed_0           1
company             1
typename            1
inches              2
screenresolution    1
cpu                 1
ram                 1
memory              2
gpu                 1
opsys               1
weight              2
price               1
dtype: int64

--- DATA SAMPLE ---


,unnamed_0,company,typename,inches,screenresolution,cpu,ram,memory,gpu,opsys,weight,price
0,0.0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,71378.6832
1,1.0,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,47895.5232
2,2.0,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,30636.0000


In [ ]:
# drop old index
clean_laptop_df = clean_laptop_df.drop(columns=['unnamed_0'])

# strip 'GB' from Ram & convert it to float
clean_laptop_df['ram'] = clean_laptop_df['ram'].str.replace('GB', '').astype(float)

# Strip 'kg' from weight and convert it to float
clean_laptop_df['weight'] = clean_laptop_df['weight'].str.replace('kg', '').astype(float)

# Rechecking after editing
print("---New data types---")
print(clean_laptop_df[['ram', 'weight']].dtypes)




---New data types---
ram       float64
weight    float64
dtype: object


In [ ]:
# Filter the df and show only rows with missing values
corrupted_data = clean_laptop_df[clean_laptop_df.isna().any(axis=1)]

#count and display the suspects
print(f"Found {len(corrupted_data)} rows with missing data.")
display(corrupted_data)

Found 4 rows with missing data.


,company,typename,inches,screenresolution,cpu,ram,memory,gpu,opsys,weight,price
20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
202,Dell,Ultrabook,13.3,Full HD 1920x1080,Intel Core i7 8550U 1.8GHz,8.0,256GB SSD,Intel UHD Graphics 620,Windows 10,NaN,77202.7200
466,Dell,Workstation,NaN,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,8.0,256GB SSD,Nvidia Quadro M1200,Windows 10,1.78,128298.2400
750,Dell,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,16.0,NaN,AMD Radeon R7 M445,Windows 10,2.30,62938.0656


In [ ]:
# drop corrupted rows and reset index
clean_laptop_df = clean_laptop_df.dropna().reset_index(drop=True)
#surviving rows
print(f"rows left: {len(clean_laptop_df)}\n")
# audit simple categories
simple_categories = ['company', 'typename', 'opsys']

for col in simple_categories:
  print(f"---{col.upper()}---")
  print(clean_laptop_df[col].value_counts())
  print()

# count complex text variations
complex_text = ['screenresolution', 'cpu', 'memory', 'gpu']

for col in complex_text:
  unique_count = clean_laptop_df[col].nunique()
  print(f"{col.upper()}: {unique_count} distinct variations")

rows left: 1270

---COMPANY---
company
Lenovo       290
Dell         284
HP           266
Asus         156
Acer         103
MSI           53
Toshiba       47
Apple         21
Samsung        9
Mediacom       7
Razer          7
Microsoft      6
Vero           4
Xiaomi         4
Chuwi          3
LG             3
Google         3
Huawei         2
Fujitsu        2
Name: count, dtype: int64

---TYPENAME---
typename
Notebook              709
Gaming                203
Ultrabook             190
2 in 1 Convertible    116
Workstation            28
Netbook                24
Name: count, dtype: int64

---OPSYS---
opsys
Windows 10      1044
No OS             63
Linux             61
Windows 7         45
Chrome OS         27
macOS             13
Mac OS X           8
Windows 10 S       8
Android            1
Name: count, dtype: int64

SCREENRESOLUTION: 40 distinct variations
CPU: 118 distinct variations
MEMORY: 39 distinct variations
GPU: 106 distinct variations


In [15]:
clean_laptop_df['opsys'] = clean_laptop_df['opsys'].replace({'Mac OS X': 'macOS'})

print(clean_laptop_df['opsys'].value_counts())

opsys
Windows 10      1044
No OS             63
Linux             61
Windows 7         45
Chrome OS         27
macOS             21
Windows 10 S       8
Android            1
Name: count, dtype: int64


In [16]:
# summary stats on prices column
print(clean_laptop_df['price'].describe())


count      1270.000000
mean      59886.072669
std       37323.784300
min        9270.720000
25%       31914.720000
50%       52054.560000
75%       79303.816800
max      324954.720000
Name: price, dtype: float64


In [17]:
max_price = clean_laptop_df['price'].max()
expensive_laptop = clean_laptop_df[clean_laptop_df['price'] == max_price]
print(expensive_laptop)

# checking if laptops have exact specs but different prices
hardware_cols = ['company', 'typename', 'ram', 'cpu', 'memory', 'gpu', 'opsys']

# find rows where the hardware is identical but appear more than once
spec_duplicates = clean_laptop_df[clean_laptop_df.duplicated(subset=hardware_cols,keep=False)]

# sort them by company and cpu so that identical laptops sit next to each other
spec_duplicates = spec_duplicates.sort_values(by=['company', 'cpu'])

print(f"\n---Found {len(spec_duplicates)} Laptops with identical specs--- ")
display(spec_duplicates.head(6))





    company typename  inches                     screenresolution  \
189   Razer   Gaming    17.3  4K Ultra HD / Touchscreen 3840x2160   

                             cpu   ram   memory                      gpu  \
189  Intel Core i7 7820HK 2.9GHz  32.0  1TB SSD  Nvidia GeForce GTX 1080   

          opsys  weight      price  
189  Windows 10    3.49  324954.72  

---Found 518 Laptops with identical specs--- 


,company,typename,inches,screenresolution,cpu,ram,memory,gpu,opsys,weight,price
803,Acer,Netbook,11.6,IPS Panel 1366x768,Intel Celeron Dual Core N3060 1.6GHz,4.0,32GB Flash Storage,Intel HD Graphics 400,Chrome OS,1.35,18914.40
863,Acer,Netbook,11.6,IPS Panel 1366x768,Intel Celeron Dual Core N3060 1.6GHz,4.0,32GB Flash Storage,Intel HD Graphics 400,Chrome OS,1.25,15824.16
1230,Acer,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4.0,500GB HDD,Intel HD Graphics 400,Linux,2.40,15397.92
1244,Acer,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4.0,500GB HDD,Intel HD Graphics 400,Linux,2.40,15397.92
1258,Acer,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4.0,500GB HDD,Intel HD Graphics 400,Linux,2.40,15397.92
448,Acer,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,4.0,1TB HDD,Nvidia GeForce 940MX,Windows 10,2.23,28984.32


In [20]:
print("Before (Raw data)")
print(f"Total rows: {len(raw_laptop_df)} | Total nulls: {raw_laptop_df.isna().sum().sum()}")
print("After (Cleaned data)")
print(f"Total rows: {len(clean_laptop_df)} | Total nulls: {clean_laptop_df.isna().sum().sum()}")
#Prove the math type changed
print(clean_laptop_df[['ram', 'weight']].dtypes)

Before (Raw data)
Total rows: 1303 | Total nulls: 360
After (Cleaned data)
Total rows: 1270 | Total nulls: 0
ram       float64
weight    float64
dtype: object
